In [1]:
#autoreload
%load_ext autoreload
%autoreload 2

In [2]:
import warcraftlogs
from warcraftlogs.constants import TOKEN_DIR

from warcraftlogs import WarcraftLogsClient

client = WarcraftLogsClient(token_dir=TOKEN_DIR)

In [ ]:
"""Main analysis function."""
# Example usage - replace with your report details
import warcraftlogs
from warcraftlogs.constants import TOKEN_DIR
from warcraftlogs import WarcraftLogsClient
import pandas as pd

client = WarcraftLogsClient(token_dir=TOKEN_DIR)
# Example report from the URL: https://www.warcraftlogs.com/reports/vDdYBKJaPVbRcz23?fight=1
#https://www.warcraftlogs.com/reports/vDdYBKJaPVbRcz23?fight=last
report_code = "vDdYBKJaPVbRcz23"
fight_id = 3

#https://www.warcraftlogs.com/reports/XCWdVDKTnkAhrZzB?fight=last&type=damage-done
from warcraftlogs.query.reports import get_last_fight_id
report_code = "XCWdVDKTnkAhrZzB"
fight_id = get_last_fight_id(client=client, report_code=report_code)

In [50]:
report_code, fight_id

('XCWdVDKTnkAhrZzB', 1)

In [4]:
from warcraftlogs.query.timeline.pull_analyzer import *

In [130]:
analyzer.actor_lookup[9]

{'id': 9,
 'name': 'Undercrawler',
 'gameID': 231380,
 'type': 'NPC',
 'subType': 'NPC'}

In [132]:
analyzer = DungeonPullAnalyzer(client)
# Load the report data
fight_info = analyzer.load_report_data(report_code, fight_id)

pulls = analyzer.analyze_comprehensive_pulls_combined(gap_threshold=15000)

Loading report XCWdVDKTnkAhrZzB, fight 1...
Loaded fight: Operation: Floodgate
Duration: 1448.6 seconds
Loaded 49 actors
Analyzing full dungeon (1449s) using COMBINED damage + threat events...
  Querying events from +0s to +180s...
    Found 2013 damage + 410 threat events
  Querying events from +180s to +360s...
    Found 2004 damage + 207 threat events
  Querying events from +360s to +540s...
    Found 2002 damage + 141 threat events
  Querying events from +540s to +720s...
    Found 2000 damage + 121 threat events
  Querying events from +720s to +900s...
    Found 2002 damage + 330 threat events
  Querying events from +900s to +1080s...
    Found 2000 damage + 19 threat events
  Querying events from +1080s to +1260s...
    Found 2003 damage + 164 threat events
  Querying events from +1260s to +1440s...
    Found 2000 damage + 37 threat events
processing gigazap events
{'timestamp': 10936345, 'type': 'damage', 'sourceID': 5, 'targetID': 46, 'abilityGameID': 195292, 'fight': 1, 'hitTy

In [133]:
game_id_to_name = {mob_instance.to_dict()['game_id']: mob_instance.to_dict()['name'] for mob_instance in analyzer.all_mob_instances}

In [134]:
from warcraftlogs.query.timeline.tankpull_analyzer import TankPullAnalyzer

In [135]:
from warcraftlogs.query.reports import get_player_dps_and_ilvl, categorize_players_by_role

In [186]:
player_dps_and_ilvl = get_player_dps_and_ilvl(
    report_code=report_code,
    fight_id=fight_id,
    query_graphql_func=client.query_public_api
)

/Users/shadowclone/Desktop/Code/warcraftlogs/warcraftlogs/query/timeline/tankpull_analyzer.py:351: SyntaxWarning: "is not" with 'int' literal. Did you mean "!="?
  tank_name=self.tank_info["tank_name"],


In [137]:
tank_name = categorize_players_by_role(player_dps_and_ilvl['players'])['tank'][0]['name']

In [138]:
print_pull_analysis(pulls, fight_info)


Detected 21 pulls using :

Pull 1:   15.3s -   56.0s ( 40.7s)
   2x Darkfuse Soldier               (instances 1-2)
   3x Loaderbot                      (instances 1-3)
   1x Shreddinator 3000              (instances 1)
   1x Undercrawler                   (instances 0)
   7x Venture Co. Contractor         (instances 1-7)
   2x Venture Co. Surveyor           (instances 1-2)

Pull 2:   71.3s -  113.9s ( 42.6s)
   4x Darkfuse Hyena                 (instances 1-4)
   2x Darkfuse Soldier               (instances 3-4)
   2x Venture Co. Contractor         (instances 8-9)
   1x Venture Co. Surveyor           (instances 3)

Pull 3:  129.7s -  167.4s ( 37.7s)
   3x Loaderbot                      (instances 4-6)
   1x Venture Co. Architect          (instances 1)
   2x Venture Co. Contractor         (instances 10-11)
   1x Venture Co. Surveyor           (instances 4)

Pull 4:  173.5s -  240.6s ( 67.2s)
   1x Scaffolding                    (instances 2)
   1x Venture Co. Architect          (instan

In [140]:
pull_dict_list = [pull.to_dict() for pull in pulls]

In [141]:
chain_pull_results = analyze_chain_pulls(detected_pulls=pulls, gap_threshold=5000)


Analyzing for chain pulls (gap threshold: 5.0s)...
Found 4 potential chain pulls:
  Pull 7 started 80.3s before Pull 6 ended
  Pull 9 started 27.6s before Pull 8 ended
  Pull 10 started 3.0s before Pull 9 ended
  Pull 14 started 27.7s before Pull 13 ended


In [196]:
tankpull_analyzer = TankPullAnalyzer(report_code, fight_id, client, game_id_to_name)

pull_analyses = tankpull_analyzer.analyze_multiple_pulls(
    pulls_data=pull_dict_list,
    chain_pull_info=chain_pull_results,
    pre_pull_seconds=15,
    post_pull_seconds=15,
)

/Users/shadowclone/Desktop/Code/warcraftlogs/warcraftlogs/query/timeline/tankpull_analyzer.py:351: SyntaxWarning: "is not" with 'int' literal. Did you mean "!="?
  cast_events = list(filter(lambda x: x.target_info.target_id in pull_target_ids if x.target_info and x.target_info.target_id is not -1 else True, cast_events))


Pull 2 time diff from previous pull: 15.3 seconds
Adjusted pre_pull_seconds for pull 2: 15.0 seconds
Pull 3 time diff from previous pull: 15.8 seconds
Adjusted pre_pull_seconds for pull 3: 15.0 seconds
Pull 4 time diff from previous pull: 6.1 seconds
Adjusted pre_pull_seconds for pull 4: 6.1 seconds
Pull 5 time diff from previous pull: 7.3 seconds
Adjusted pre_pull_seconds for pull 5: 7.3 seconds
Pull 6 time diff from previous pull: 7.1 seconds
Adjusted pre_pull_seconds for pull 6: 7.1 seconds
Pull 7 time diff from previous pull: -80.3 seconds
Pull 8 time diff from previous pull: 39.7 seconds
Adjusted pre_pull_seconds for pull 8: 15.0 seconds
Pull 9 time diff from previous pull: -27.6 seconds
Pull 10 time diff from previous pull: -3.0 seconds
Pull 11 time diff from previous pull: 16.2 seconds
Adjusted pre_pull_seconds for pull 11: 15.0 seconds
Pull 12 time diff from previous pull: 8.7 seconds
Adjusted pre_pull_seconds for pull 12: 8.7 seconds
Pull 13 time diff from previous pull: 12.6 

In [197]:
from warcraftlogs.query.timeline.tankpull_analyzer import format_tank_pull_analysis

In [199]:
chain_pull_results

[{'pull_id': 7,
  'prev_pull_id': 6,
  'overlap_duration': 80.348,
  'gap_time': -80.348},
 {'pull_id': 9,
  'prev_pull_id': 8,
  'overlap_duration': 27.575,
  'gap_time': -27.575},
 {'pull_id': 10,
  'prev_pull_id': 9,
  'overlap_duration': 2.984,
  'gap_time': -2.984},
 {'pull_id': 14,
  'prev_pull_id': 13,
  'overlap_duration': 27.734,
  'gap_time': -27.734}]

In [198]:
for pull_info in pull_analyses:
    if pull_info.pull_info.is_chain_pull is False:
        print(format_tank_pull_analysis(pull_info, show_all_casts=True))
        print("\n\n\n\n")

════════════════════════════════════════════════════════════════════════════════
🛡️  TANK PULL ANALYSIS - PULL 1
════════════════════════════════════════════════════════════════════════════════
👤 Tank: Nazuqi (Blood DeathKnight)
📊 Report: XCWdVDKTnkAhrZzB | Fight: 1
⏱️  Pull Duration: 40.7 seconds
🎯 STANDARD PULL
🔍 Analysis Window: -15s to +15s from pull start

🏹 MOBS IN PULL (16 total):
   • Darkfuse Soldier (ID: 228144) - Instances: 1, 2
   • Venture Co. Contractor (ID: 229250) - Instances: 1, 2, 3, 4, 5, 6, 7
   • Undercrawler (ID: 231380) - Instance #0
   • Shreddinator 3000 (ID: 230740) - Instance #1
   • Loaderbot (ID: 231014) - Instances: 1, 2, 3
   • Venture Co. Surveyor (ID: 229686) - Instances: 1, 2

📋 PRE-PULL PREPARATION (1 casts):
     -0.5s: Death's Caress → Darkfuse Soldier#1

⚔️  PULL EXECUTION (15 total casts):
     +0.2s: Death's Advance → Area/Self
     +0.7s: Marrowrend → Darkfuse Soldier#1
     +1.9s: Blood Boil → Area/Self
     +3.5s: Algari Healing Potion → Area/

In [194]:
for pull_info in pull_analyses:
    if pull_info.pull_info.is_chain_pull is False:
        print(format_tank_pull_analysis(pull_info, show_all_casts=True))
        print("\n\n\n\n")

════════════════════════════════════════════════════════════════════════════════
🛡️  TANK PULL ANALYSIS - PULL 1
════════════════════════════════════════════════════════════════════════════════
👤 Tank: Nazuqi (Blood DeathKnight)
📊 Report: XCWdVDKTnkAhrZzB | Fight: 1
⏱️  Pull Duration: 40.7 seconds
🎯 STANDARD PULL
🔍 Analysis Window: -15s to +15s from pull start

🏹 MOBS IN PULL (16 total):
   • Darkfuse Soldier (ID: 228144) - Instances: 1, 2
   • Venture Co. Contractor (ID: 229250) - Instances: 1, 2, 3, 4, 5, 6, 7
   • Undercrawler (ID: 231380) - Instance #0
   • Shreddinator 3000 (ID: 230740) - Instance #1
   • Loaderbot (ID: 231014) - Instances: 1, 2, 3
   • Venture Co. Surveyor (ID: 229686) - Instances: 1, 2

📋 PRE-PULL PREPARATION (1 casts):
     -0.5s: Death's Caress → Darkfuse Soldier#1

⚔️  PULL EXECUTION (15 total casts):
     +0.2s: Death's Advance → Area/Self
     +0.7s: Marrowrend → Darkfuse Soldier#1
     +1.9s: Blood Boil → Area/Self
     +3.5s: Algari Healing Potion → Area/